In [5]:
import scipy.io

# 选择您上传的任意一个文件名
mat_file_to_inspect = '../data/raw/target/A.mat' 

try:
    data = scipy.io.loadmat(mat_file_to_inspect)
    print(f"--- 文件 '{mat_file_to_inspect}' 内部变量名列表 ---")
    
    # 打印文件中所有变量名 (keys)
    keys = [key for key in data.keys() if not key.startswith('__')]
    print(keys)
    
except FileNotFoundError:
    print(f"错误：请确保 '{mat_file_to_inspect}' 文件与此脚本在同一个文件夹下。")
except Exception as e:
    print(f"读取文件时出错: {e}")

--- 文件 'A.mat' 内部变量名列表 ---
['A']


In [7]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import scipy.io
import os
import glob
from tqdm import tqdm

# =============================================================================
# 1. 特征提取核心函数
# =============================================================================
def calculate_time_features(signal: pd.Series) -> dict:
    """计算一维信号的时域特征"""
    signal = signal.dropna()
    rms = np.sqrt(np.mean(signal**2))
    sqrt_amp = (np.mean(np.sqrt(np.abs(signal))))**2
    
    features = {
        'Mean': np.mean(signal), 'RMS': rms, 'Var': np.var(signal),
        'Skew': skew(signal), 'Kurt': kurtosis(signal),
        'CF': np.max(np.abs(signal)) / rms if rms != 0 else 0,
        'MF': np.max(np.abs(signal)) / sqrt_amp if sqrt_amp != 0 else 0,
        'P2P': np.max(signal) - np.min(signal),
    }
    return features

def calculate_freq_features(signal: pd.Series, sampling_rate: int) -> dict:
    """计算一维信号的频域特征"""
    signal = signal.dropna()
    n_points = len(signal)
    if n_points == 0:
        return {'FreqMean': 0, 'FreqSTD': 0, 'FreqSkew': 0, 'FreqKurt': 0}

    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_freq_indices = np.where(fft_freq >= 0)
    
    # --- **核心修正**：使用正确的变量名 positive_freq_indices ---
    freqs = fft_freq[positive_freq_indices]
    amplitudes = np.abs(fft_vals[positive_freq_indices])
    
    power_spectrum = amplitudes**2
    power_spectrum_normalized = power_spectrum / np.sum(power_spectrum) if np.sum(power_spectrum) != 0 else power_spectrum
    
    freq_mean = np.sum(freqs * power_spectrum_normalized)
    freq_std = np.sqrt(np.sum(((freqs - freq_mean)**2) * power_spectrum_normalized))
    epsilon = 1e-10
    freq_skew = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**3 * power_spectrum_normalized)
    freq_kurt = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**4 * power_spectrum_normalized) - 3

    features = {
        'FreqMean': freq_mean, 'FreqSTD': freq_std,
        'FreqSkew': freq_skew, 'FreqKurt': freq_kurt
    }
    return features

# =============================================================================
# 2. 主程序：遍历、处理、整合
# =============================================================================
def process_flat_directory_auto_key(root_dir: str, sampling_rate: int = 12000):
    """
    遍历指定的单个文件夹，根据文件名自动推断变量名，并提取特征。
    """
    if not os.path.isdir(root_dir):
        print(f"❌ 错误：提供的路径 '{root_dir}' 不是一个有效的文件夹。")
        return

    mat_files = sorted(glob.glob(os.path.join(root_dir, '*.mat')))
    if not mat_files:
        print(f"❌ 在文件夹 '{root_dir}' 中未找到任何 .mat 文件。")
        return

    print(f"✅ 在文件夹 '{root_dir}' 中找到 {len(mat_files)} 个 .mat 文件，开始处理...")
    
    all_features_list = []
    
    for file_path in tqdm(mat_files, desc="Processing .mat files"):
        try:
            base_name = os.path.basename(file_path)
            signal_key, _ = os.path.splitext(base_name)
            
            data = scipy.io.loadmat(file_path)
            
            if signal_key in data:
                signal = pd.Series(data[signal_key].flatten())
                
                time_feats = calculate_time_features(signal)
                freq_feats = calculate_freq_features(signal, sampling_rate)
                
                all_feats = {**time_feats, **freq_feats}
                all_feats['SourceFile'] = base_name
                all_feats['Name'] = signal_key
                
                all_features_list.append(all_feats)
            else:
                print(f"\n警告: 在文件 '{base_name}' 中未找到预期的变量名 '{signal_key}'，已跳过。")

        except Exception as e:
            print(f"\n警告: 处理文件 {os.path.basename(file_path)} 时出错: {e}")

    if not all_features_list:
        print("\n扫描完成，但未能从任何文件中提取有效特征。请检查文件名和变量名是否匹配。")
        return

    final_df = pd.DataFrame(all_features_list)
    id_cols = ['SourceFile', 'Name']
    feature_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + feature_cols]
    
    output_filename = '../data/features/target/extracted_features_auto.csv'
    final_df.to_csv(output_filename, index=False)
    
    print(f"\n\n🎉 --- 处理完成 --- 🎉")
    print(f"所有特征已汇总并保存到文件: '{output_filename}'")
    print("\n最终输出数据预览:")
    print(final_df.head().to_string())


# --- 如何使用 ---
if __name__ == "__main__":
    # ========================= 用户配置区 =========================
    
    # 1. 指定存放所有.mat文件的【单个文件夹】的路径
    TARGET_DIRECTORY = '../data/raw/target' 
    
    # 2. 设定正确的采样率
    SAMPLING_RATE = 32000
    
    # ======================= 配置结束 ===========================
    
    process_flat_directory_auto_key(
        root_dir=TARGET_DIRECTORY, 
        sampling_rate=SAMPLING_RATE
    )

✅ 在文件夹 '.' 中找到 16 个 .mat 文件，开始处理...


Processing .mat files: 100%|██████████| 16/16 [00:00<00:00, 25.74it/s]



🎉 --- 处理完成 --- 🎉
所有特征已汇总并保存到文件: 'extracted_features_auto.csv'

最终输出数据预览:
  SourceFile Name      Mean        RMS         Var      Skew       Kurt         CF         MF         P2P     FreqMean      FreqSTD  FreqSkew  FreqKurt
0      A.mat    A  0.000297   3.434284   11.794305  0.010704   0.156264   4.662128   6.960216   31.204360  4189.408114  2727.889377  0.710584 -0.118648
1      B.mat    B  0.000330   1.468095    2.155303  0.032511   1.704747   9.152530  14.309264   24.924153  5196.133075  3278.898853  0.525445  0.002382
2      C.mat    C -0.003674  31.522846  993.689785  0.070316  25.287749  11.087238  31.528812  698.195901  7749.357853  2359.209994  0.213565  2.071755
3      D.mat    D -0.001776   8.286112   68.659652  0.013287   0.068857   4.567380   6.780505   72.011691  5963.553869  2000.232824 -0.516904  0.376493
4      E.mat    E -0.011236  15.665019  245.392691 -0.047967  37.933641  11.488510  39.817136  359.807106  7730.598593  2726.057342 -0.509413  1.138385


In [1]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import skew, kurtosis
import os
import glob
from tqdm import tqdm

# =============================================================================
# 1. 特征提取核心函数 (已更新)
# =============================================================================
def calculate_all_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有需要的基础和高级特征。
    已根据您的ex.ipynb补全所有时域指标。
    """
    # --- 基础时域特征 ---
    n = len(signal)
    mean_abs = np.mean(np.abs(signal))
    rms = np.sqrt(np.mean(signal**2))
    
    # **【新增】** 根据您的ex.ipynb补全的指标
    shape_factor = rms / mean_abs if mean_abs != 0 else 0
    impulse_factor = np.max(np.abs(signal)) / mean_abs if mean_abs != 0 else 0
    clearance_factor = np.max(np.abs(signal)) / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    
    # **【更新】** 裕度因子和峰值因子的计算方式与您的ex.ipynb保持一致
    crest_factor = np.max(np.abs(signal)) / rms if rms != 0 else 0
    margin_factor = clearance_factor # 在您的ex.ipynb中，裕度因子和峭度因子计算方式相同
    
    all_features = {
        'Mean': np.mean(signal),
        'Mean_abs': mean_abs, # 新增
        'Var': np.var(signal),
        'Std': np.std(signal), # 新增
        'Kurt': kurtosis(signal),
        'Skew': skew(signal),
        'RMS': rms,
        'Crest': crest_factor, # 更新变量名
        'Shape': shape_factor, # 新增
        'Impulse': impulse_factor, # 新增
        'Margin': margin_factor, # 更新变量名
        'Clearance': clearance_factor, # 新增
        'P2P': np.max(signal) - np.min(signal)
    }

    # --- 高级频域特征 (逻辑保持不变) ---
    n_points = len(signal)
    fr = rpm / 60.0
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # Part 1: 边带分析
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])

    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        all_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        all_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        all_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        all_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # Part 2: 包络解调分析
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return all_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        all_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        all_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        all_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return all_features

# =============================================================================
# 2. 主执行函数 (逻辑保持不变)
# =============================================================================
def process_flat_directory(root_dir: str, rpm_value: float, sampling_rate: int):
    """
    遍历指定的单个文件夹，使用固定的RPM值，并提取所有特征。
    """
    if not os.path.isdir(root_dir):
        print(f"❌ 错误：提供的路径 '{root_dir}' 不是一个有效的文件夹。")
        return

    mat_files = sorted(glob.glob(os.path.join(root_dir, '*.mat')))
    if not mat_files:
        print(f"❌ 在文件夹 '{root_dir}' 中未找到任何 .mat 文件。")
        return

    print(f"✅ 在文件夹 '{root_dir}' 中找到 {len(mat_files)} 个 .mat 文件，开始处理...")
    
    all_results = []
    
    for file_path in tqdm(mat_files, desc="Processing .mat files"):
        try:
            base_name = os.path.basename(file_path)
            signal_key, _ = os.path.splitext(base_name)
            
            data = scipy.io.loadmat(file_path)
            
            if signal_key in data:
                signal = data[signal_key].flatten()
                
                features = calculate_all_features(signal, rpm_value, sampling_rate)
                
                features['SourceFile'] = base_name
                features['RPM'] = rpm_value
                
                all_results.append(features)
            else:
                print(f"\n警告: 在文件 '{base_name}' 中未找到预期的变量名 '{signal_key}'，已跳过。")

        except Exception as e:
            print(f"\n警告: 处理文件 {os.path.basename(file_path)} 时出错: {e}")

    if not all_results:
        print("\n扫描完成，但未能从任何文件中提取有效特征。")
        return

    final_df = pd.DataFrame(all_results)
    
    id_cols = ['SourceFile', 'RPM']
    feature_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + feature_cols]
    
    output_filename = '../data/features/target/final_features_complete.csv'
    final_df.to_csv(output_filename, index=False)
    
    print(f"\n\n🎉 --- 处理完成 --- 🎉")
    print(f"所有特征已汇总并保存到文件: '{output_filename}'")
    print("\n最终输出数据预览:")
    print(final_df.head().to_string())


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    # 1. 指定存放所有.mat文件的【单个文件夹】的路径
    TARGET_DIRECTORY = '../data/raw/target' 
    
    # 2. 设置用于所有文件的【固定RPM值】
    RPM_VALUE = 600
    
    # 3. 设定正确的采样率
    SAMPLING_RATE = 32000
    
    # ======================= 配置结束 ===========================
    
    process_flat_directory(
        root_dir=TARGET_DIRECTORY,
        rpm_value=RPM_VALUE, 
        sampling_rate=SAMPLING_RATE
    )

✅ 在文件夹 '.' 中找到 16 个 .mat 文件，开始处理...


Processing .mat files: 100%|██████████| 16/16 [00:01<00:00, 13.40it/s]



🎉 --- 处理完成 --- 🎉
所有特征已汇总并保存到文件: 'final_features_complete.csv'

最终输出数据预览:
  SourceFile  RPM      Mean   Mean_abs         Var        Std       Kurt      Skew        RMS      Crest     Shape    Impulse     Margin  Clearance         P2P  IR_Sideband_Energy_DE  IR_Sideband_Ratio_DE  B_Sideband_Energy_DE  B_Sideband_Ratio_DE  IR_Sideband_Energy_FE  IR_Sideband_Ratio_FE  B_Sideband_Energy_FE  B_Sideband_Ratio_FE  Env_Peak_BPFO_DE  Env_Peak_BPFI_DE  Env_Peak_BSF_DE  Env_Peak_BPFO_FE  Env_Peak_BPFI_FE  Env_Peak_BSF_FE
0      A.mat  600  0.000297   2.723423   11.794305   3.434284   0.156264  0.010704   3.434284   4.662128  1.261017   5.879024   6.960216   6.960216   31.204360           1.478788e+07              3.766217          6.434916e+07           470.912847           1.574795e+07              2.959028          5.661934e+05             4.727399       1578.568130       2957.501967      2547.418220       2443.203650       2501.148397      1735.008003
1      B.mat  600  0.000330   1.126563   

In [3]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import skew, kurtosis
import os
import glob
from tqdm import tqdm

# =============================================================================
# 1. 特征提取核心函数 (已更新)
# =============================================================================
def calculate_all_features(signal: np.ndarray, rpm: float, sampling_rate: int) -> dict:
    """
    为单条信号计算所有需要的基础和高级特征。
    """
    # --- 基础时域特征 ---
    n = len(signal)
    mean_abs = np.mean(np.abs(signal))
    rms = np.sqrt(np.mean(signal**2))
    
    shape_factor = rms / mean_abs if mean_abs != 0 else 0
    impulse_factor = np.max(np.abs(signal)) / mean_abs if mean_abs != 0 else 0
    clearance_factor = np.max(np.abs(signal)) / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    crest_factor = np.max(np.abs(signal)) / rms if rms != 0 else 0
    margin_factor = clearance_factor
    
    all_features = {
        'Mean': np.mean(signal), 'Mean_abs': mean_abs, 'Var': np.var(signal),
        'Std': np.std(signal), 'Kurt': kurtosis(signal), 'Skew': skew(signal),
        'RMS': rms, 'Crest': crest_factor, 'Shape': shape_factor,
        'Impulse': impulse_factor, 'Margin': margin_factor, 'Clearance': clearance_factor,
        'P2P': np.max(signal) - np.min(signal)
    }

    # --- 频域特征 ---
    n_points = len(signal)
    fr = rpm / 60.0
    
    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_indices]
    amplitudes = np.abs(fft_vals[positive_indices])
    
    # **【新增】: 计算您需要的4个基础频域特征**
    power_spectrum = amplitudes**2
    power_spectrum_normalized = power_spectrum / np.sum(power_spectrum) if np.sum(power_spectrum) != 0 else power_spectrum
    
    freq_mean = np.sum(freqs * power_spectrum_normalized)
    freq_std = np.sqrt(np.sum(((freqs - freq_mean)**2) * power_spectrum_normalized))
    epsilon = 1e-10
    freq_skew = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**3 * power_spectrum_normalized)
    freq_kurt = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**4 * power_spectrum_normalized) - 3

    all_features['FreqMean'] = freq_mean
    all_features['FreqSTD'] = freq_std
    all_features['FreqSkew'] = freq_skew
    all_features['FreqKurt'] = freq_kurt
    
    # --- 高级频域特征 ---
    bearings = {
        'DE': {'n': 9, 'd': 0.3126, 'D': 1.537},
        'FE': {'n': 9, 'd': 0.2656, 'D': 1.122}
    }
    
    theoretical_freqs = {}
    for name, p in bearings.items():
        bpfo = 0.5 * p['n'] * fr * (1 - (p['d'] / p['D']))
        bpfi = 0.5 * p['n'] * fr * (1 + (p['d'] / p['D']))
        bsf = 0.5 * (p['D'] / p['d']) * fr * (1 - (p['d'] / p['D'])**2)
        ftf = 0.5 * fr * (1 - (p['d'] / p['D']))
        theoretical_freqs[name] = {'BPFO': bpfo, 'BPFI': bpfi, 'BSF': bsf, 'FTF': ftf}

    # Part 1: 边带分析
    def get_energy_at(target_freq, freq_axis, amps, window=5):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        start = max(0, idx - window)
        end = min(len(amps) - 1, idx + window)
        return np.sum(amps[start:end+1]**2)

    for name, freqs_dict in theoretical_freqs.items():
        ir_carrier_e = get_energy_at(freqs_dict['BPFI'], freqs, amplitudes)
        ir_sideband_e = get_energy_at(freqs_dict['BPFI'] - fr, freqs, amplitudes) + \
                        get_energy_at(freqs_dict['BPFI'] + fr, freqs, amplitudes)
        all_features[f'IR_Sideband_Energy_{name}'] = ir_sideband_e
        all_features[f'IR_Sideband_Ratio_{name}'] = ir_sideband_e / (ir_carrier_e + 1e-9)
        
        b_carrier_e = get_energy_at(freqs_dict['BSF'], freqs, amplitudes)
        b_sideband_e = get_energy_at(freqs_dict['BSF'] - freqs_dict['FTF'], freqs, amplitudes) + \
                       get_energy_at(freqs_dict['BSF'] + freqs_dict['FTF'], freqs, amplitudes)
        all_features[f'B_Sideband_Energy_{name}'] = b_sideband_e
        all_features[f'B_Sideband_Ratio_{name}'] = b_sideband_e / (b_carrier_e + 1e-9)

    # Part 2: 包络解调分析
    nyquist = 0.5 * sampling_rate
    low = 2000 / nyquist
    high = (nyquist - 10) / nyquist
    if low >= high: return all_features

    b, a = butter(4, [low, high], btype='band')
    filtered_signal = filtfilt(b, a, signal)
    envelope = np.abs(hilbert(filtered_signal))
    
    env_fft_vals = np.fft.fft(envelope - np.mean(envelope))
    env_fft_freq = np.fft.fftfreq(len(envelope), 1.0 / sampling_rate)
    env_positive_indices = np.where(env_fft_freq >= 0)
    env_freqs = env_fft_freq[env_positive_indices]
    env_amplitudes = np.abs(env_fft_vals[env_positive_indices])
    
    def get_peak_at(target_freq, freq_axis, amps):
        idx = np.argmin(np.abs(freq_axis - target_freq))
        return amps[idx]

    for name, freqs_dict in theoretical_freqs.items():
        all_features[f'Env_Peak_BPFO_{name}'] = get_peak_at(freqs_dict['BPFO'], env_freqs, env_amplitudes)
        all_features[f'Env_Peak_BPFI_{name}'] = get_peak_at(freqs_dict['BPFI'], env_freqs, env_amplitudes)
        all_features[f'Env_Peak_BSF_{name}'] = get_peak_at(freqs_dict['BSF'], env_freqs, env_amplitudes)

    return all_features

# =============================================================================
# 2. 主执行函数 (逻辑保持不变)
# =============================================================================
def process_flat_directory(root_dir: str, rpm_value: float, sampling_rate: int):
    """
    遍历指定的单个文件夹，使用固定的RPM值，并提取所有特征。
    """
    if not os.path.isdir(root_dir):
        print(f"❌ 错误：提供的路径 '{root_dir}' 不是一个有效的文件夹。")
        return

    mat_files = sorted(glob.glob(os.path.join(root_dir, '*.mat')))
    if not mat_files:
        print(f"❌ 在文件夹 '{root_dir}' 中未找到任何 .mat 文件。")
        return

    print(f"✅ 在文件夹 '{root_dir}' 中找到 {len(mat_files)} 个 .mat 文件，开始处理...")
    
    all_results = []
    
    for file_path in tqdm(mat_files, desc="Processing .mat files"):
        try:
            base_name = os.path.basename(file_path)
            signal_key, _ = os.path.splitext(base_name)
            
            data = scipy.io.loadmat(file_path)
            
            if signal_key in data:
                signal = data[signal_key].flatten()
                
                features = calculate_all_features(signal, rpm_value, sampling_rate)
                
                features['SourceFile'] = base_name
                features['RPM'] = rpm_value
                
                all_results.append(features)
            else:
                print(f"\n警告: 在文件 '{base_name}' 中未找到预期的变量名 '{signal_key}'，已跳过。")

        except Exception as e:
            print(f"\n警告: 处理文件 {os.path.basename(file_path)} 时出错: {e}")

    if not all_results:
        print("\n扫描完成，但未能从任何文件中提取有效特征。")
        return

    final_df = pd.DataFrame(all_results)
    
    id_cols = ['SourceFile', 'RPM']
    feature_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + feature_cols]
    
    output_filename = '../data/features/target/final_features_complete.csv'
    final_df.to_csv(output_filename, index=False)
    
    print(f"\n\n🎉 --- 处理完成 --- 🎉")
    print(f"所有特征已汇总并保存到文件: '{output_filename}'")
    print("\n最终输出数据预览:")
    print(final_df.head().to_string())


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    TARGET_DIRECTORY = '../data/raw/target' 
    RPM_VALUE = 600
    SAMPLING_RATE = 32000
    
    # ======================= 配置结束 ===========================
    
    process_flat_directory(
        root_dir=TARGET_DIRECTORY,
        rpm_value=RPM_VALUE, 
        sampling_rate=SAMPLING_RATE
    )

✅ 在文件夹 '.' 中找到 16 个 .mat 文件，开始处理...


Processing .mat files: 100%|██████████| 16/16 [00:01<00:00, 14.29it/s]



🎉 --- 处理完成 --- 🎉
所有特征已汇总并保存到文件: 'final_features_complete.csv'

最终输出数据预览:
  SourceFile  RPM      Mean   Mean_abs         Var        Std       Kurt      Skew        RMS      Crest     Shape    Impulse     Margin  Clearance         P2P     FreqMean      FreqSTD  FreqSkew  FreqKurt  IR_Sideband_Energy_DE  IR_Sideband_Ratio_DE  B_Sideband_Energy_DE  B_Sideband_Ratio_DE  IR_Sideband_Energy_FE  IR_Sideband_Ratio_FE  B_Sideband_Energy_FE  B_Sideband_Ratio_FE  Env_Peak_BPFO_DE  Env_Peak_BPFI_DE  Env_Peak_BSF_DE  Env_Peak_BPFO_FE  Env_Peak_BPFI_FE  Env_Peak_BSF_FE
0      A.mat  600  0.000297   2.723423   11.794305   3.434284   0.156264  0.010704   3.434284   4.662128  1.261017   5.879024   6.960216   6.960216   31.204360  4189.408114  2727.889377  0.710584 -0.118648           1.478788e+07              3.766217          6.434916e+07           470.912847           1.574795e+07              2.959028          5.661934e+05             4.727399       1578.568130       2957.501967      2547.418220   

注：此处进行了一些人工调整，无需再运行Feature Engineering，直接可以进入下一步